In [1]:
from langchain_groq import ChatGroq

In [2]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='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', 
    model_name="llama3-70b-8192"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

That's an easy one!

The first person to set foot on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to walk on the moon.


In [3]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/director-software-engineering/job/R-56205")
page_data = loader.load().pop().page_content
print(page_data)






















Director- Software Engineering










































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Men

In [5]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [6]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

[{'role': 'Director- Software Engineering',
  'experience': '12+ years of technology experience in software development and management of full stack software solutions with hands-on software engineering experience',
  'skills': ['Agile development practices',
   'Software architecture, design and DevOps principles',
   'Building scalable platforms using micro service architecture, domain driven design and RESTful Services using languages such as Java, Nodejs, and typescript',
   'Modern UI frameworks (reactJs, nextjs etc), test frameworks, cloud services, preferrably AWS, Dev Ops techniques and tools',
   'AI/ML and GenAI features'],
  'description': "We are seeking a driven engineering leader to join our team. As the Director of Engineering of Marketing Assets Platform, you'll be responsible for leading media asset management and delivery of assets that power all of Nike’s consumer experiences in marketplaces globally."}]

In [7]:
type(json_res)


list

In [8]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [10]:
import uuid
import chromadb
import pandas as pd

# Assuming df is already defined (with columns "Techstack" and "Links")
# Example:
# df = pd.read_csv("your_file.csv")

client = chromadb.PersistentClient(path="vectorstore")
collection = client.get_or_create_collection(name="portfolio")

# Only add if collection is empty
if collection.count() == 0:
    for _, row in df.iterrows():
        collection.add(
            documents=[row["Techstack"]],
            metadatas=[{"links": row["Links"]}],
            ids=[str(uuid.uuid4())]
        )


C:\Users\MBX2KOR\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|████| 79.3M/79.3M [04:53<00:00, 283kiB/s]


In [13]:
links = collection.query(query_texts=json_res[0]['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/devops-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/xamarin-portfolio'},
  {'links': 'https://example.com/full-stack-js-portfolio'}],
 [{'links': 'https://example.com/react-portfolio'},
  {'links': 'https://example.com/full-stack-js-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}]]

In [14]:
json_res

[{'role': 'Director- Software Engineering',
  'experience': '12+ years of technology experience in software development and management of full stack software solutions with hands-on software engineering experience',
  'skills': ['Agile development practices',
   'Software architecture, design and DevOps principles',
   'Building scalable platforms using micro service architecture, domain driven design and RESTful Services using languages such as Java, Nodejs, and typescript',
   'Modern UI frameworks (reactJs, nextjs etc), test frameworks, cloud services, preferrably AWS, Dev Ops techniques and tools',
   'AI/ML and GenAI features'],
  'description': "We are seeking a driven engineering leader to join our team. As the Director of Engineering of Marketing Assets Platform, you'll be responsible for leading media asset management and delivery of assets that power all of Nike’s consumer experiences in marketplaces globally."}]

In [16]:
job = json_res[0]
job['skills']

['Agile development practices',
 'Software architecture, design and DevOps principles',
 'Building scalable platforms using micro service architecture, domain driven design and RESTful Services using languages such as Java, Nodejs, and typescript',
 'Modern UI frameworks (reactJs, nextjs etc), test frameworks, cloud services, preferrably AWS, Dev Ops techniques and tools',
 'AI/ML and GenAI features']

In [19]:
from langchain.prompts import PromptTemplate

prompt_email = PromptTemplate.from_template(
    """
    ### ROLE REQUIREMENTS:
    {job_description}
    
    ### TASK:
    You are Mohan, a Business Development Executive at AtliQ — an AI and Software Consulting firm committed to enhancing 
    business workflows through intelligent automation solutions.
    With a proven track record, AtliQ has supported numerous businesses in scaling operations, improving efficiency, 
    streamlining processes, and lowering costs through tailored technology strategies.
    
    Your objective is to craft a compelling cold email to the client regarding the opportunity listed above. The email 
    should highlight how AtliQ’s services align with their needs and demonstrate our capabilities.
    Incorporate the most relevant references from the following portfolio links to strengthen the message: {link_list}
    
    Stay in character as Mohan, BDE at AtliQ. Avoid any introductory commentary or explanations.
    
    ### COLD EMAIL (START DIRECTLY):
    
    """
)

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)


Subject: Expert Software Engineering Leadership for Nike's Marketing Assets Platform

Dear Hiring Manager,

I came across the Director-Software Engineering role at Nike, and I'm excited to introduce AtliQ, a trusted AI and Software Consulting firm, as a potential partner to support your Marketing Assets Platform. With 12+ years of technology experience, our team is well-equipped to help you lead media asset management and delivery of assets that power Nike's consumer experiences globally.

At AtliQ, we've successfully supported numerous businesses in scaling operations, improving efficiency, and streamlining processes through tailored technology strategies. Our expertise in Agile development practices, software architecture, design, and DevOps principles aligns perfectly with your requirements. We've developed scalable platforms using microservice architecture, domain-driven design, and RESTful Services using languages such as Java, Nodejs, and TypeScript.

Our portfolio showcases our 